In [2]:
# Cell 1 — Mount Drive, install packages, create folders
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PhishGuard_v2'

import os, subprocess

for folder in ['data', 'data/checkpoints', 'features', 'models', 'evaluation']:
    os.makedirs(f'{DRIVE_BASE}/{folder}', exist_ok=True)

result = subprocess.run(
    ['pip', 'install', 'networkx==3.3', 'joblib==1.4.0', 'requests==2.31.0', '-q'],
    check=False, capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'Install done')

assert os.path.exists(f'{DRIVE_BASE}/data/raw_wallet_data.csv'), \
    'raw_wallet_data.csv not found'

print('Cell 1 ready. NOW: Runtime > Restart session, then re-run Cell 1 and continue.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Install done
Cell 1 ready. NOW: Runtime > Restart session, then re-run Cell 1 and continue.


In [3]:
# Cell 2 — Define load_scam_set and extract_graph_features
import json, requests, os
import networkx as nx
import numpy as np

def load_scam_set(drive_base):
    scam_address_set = set()
    path = f'{drive_base}/data/scam_addresses.json'
    if not os.path.exists(path):
        print(f'WARNING: {path} not found — run Cell 2b first')
        print('WARNING: Scam set is empty — avg_shortest_path_to_known_scam '
              'will be 99.0 for all rows.')
        return scam_address_set
    try:
        with open(path) as f:
            data = json.load(f)
        for item in data:
            if isinstance(item, str) and item.startswith('0x'):
                scam_address_set.add(item.lower())
    except Exception as e:
        print(f'WARNING: Could not parse {path}: {e}')
    print(f'Scam set loaded: {len(scam_address_set):,} addresses')
    if len(scam_address_set) == 0:
        print('WARNING: Scam set is empty — avg_shortest_path_to_known_scam '
              'will be 99.0 for all rows.')
    return scam_address_set

def extract_graph_features(txs, address, scam_address_set):
    defaults = {
        'pagerank_subgraph':               0.0,
        'clustering_coefficient':          0.0,
        'reciprocity_ratio':               0.0,
        'avg_shortest_path_to_known_scam': 99.0
    }
    try:
        addr   = address.lower()
        recent = [t for t in txs[:100] if t.get('isError') == '0']
        if not recent:
            return defaults

        G = nx.DiGraph()
        for tx in recent:
            frm = tx.get('from', '').lower()
            to  = tx.get('to',   '').lower()
            if frm and to:
                weight = int(tx.get('value', 0)) / 1e18
                G.add_edge(frm, to, weight=weight)

        if G.number_of_edges() == 0 or addr not in G.nodes():
            return defaults

        try:
            pr       = nx.pagerank(G, alpha=0.85)
            pagerank = pr.get(addr, 0.0)
        except Exception:
            pagerank = 0.0

        try:
            G_und      = G.to_undirected()
            cc         = nx.clustering(G_und)
            clustering = cc.get(addr, 0.0)
        except Exception:
            clustering = 0.0

        try:
            successors   = set(G.successors(addr))
            predecessors = set(G.predecessors(addr))
            mutual       = successors & predecessors
            all_nb       = successors | predecessors
            reciprocity  = len(mutual) / len(all_nb) if len(all_nb) > 0 else 0.0
        except Exception:
            reciprocity  = 0.0

        try:
            G_und      = G.to_undirected()
            scam_nodes = [n for n in G_und.nodes() if n in scam_address_set]
            if not scam_nodes:
                avg_scam_path = 99.0
            else:
                paths = []
                for scam_node in scam_nodes:
                    try:
                        length = nx.shortest_path_length(G_und, addr, scam_node)
                        paths.append(length)
                    except (nx.NetworkXNoPath, nx.NodeNotFound):
                        pass
                avg_scam_path = float(np.mean(paths)) if paths else 99.0
        except Exception:
            avg_scam_path = 99.0

        return {
            'pagerank_subgraph':               pagerank,
            'clustering_coefficient':          clustering,
            'reciprocity_ratio':               reciprocity,
            'avg_shortest_path_to_known_scam': avg_scam_path
        }
    except Exception:
        return defaults

print('Cell 2 functions defined.')


Cell 2 functions defined.


In [4]:
# Cell 2b — Fetch and save scam addresses (run once only)
import requests, json, os, re

SCAM_PATH = f'{DRIVE_BASE}/data/scam_addresses.json'

if os.path.exists(SCAM_PATH):
    with open(SCAM_PATH) as f:
        existing = json.load(f)
    print(f'scam_addresses.json already on Drive ({len(existing):,} addresses) — skipping download.')
    print('Delete it manually if you want to refresh.')
else:
    print('Fetching scam addresses from live sources...')
    scam_address_set = set()

    # Source 1: CryptoScamDB API
    try:
        r = requests.get('https://api.cryptoscamdb.org/v1/addresses', timeout=15)
        if r.status_code == 200:
            data = r.json()
            if data.get('success') and 'result' in data:
                before = len(scam_address_set)
                for addr in data['result'].keys():
                    if addr.startswith('0x'):
                        scam_address_set.add(addr.lower())
                print(f'Source 1 — CryptoScamDB API:      +{len(scam_address_set)-before:,} addresses')
            else:
                print('Source 1 — CryptoScamDB API:       unexpected format, skipping')
        else:
            print(f'Source 1 — CryptoScamDB API:       HTTP {r.status_code}, skipping')
    except Exception as e:
        print(f'Source 1 — CryptoScamDB API:       failed ({e})')

    # Source 2: CryptoScamDB blacklist YAML (GitHub)
    try:
        r = requests.get(
            'https://raw.githubusercontent.com/CryptoScamDB/blacklist/master/data/urls.yaml',
            timeout=15)
        if r.status_code == 200:
            before = len(scam_address_set)
            for addr in re.findall(r'"(0x[0-9a-fA-F]{40})"', r.text):
                scam_address_set.add(addr.lower())
            print(f'Source 2 — CryptoScamDB blacklist: +{len(scam_address_set)-before:,} addresses')
        else:
            print(f'Source 2 — CryptoScamDB blacklist: HTTP {r.status_code}, skipping')
    except Exception as e:
        print(f'Source 2 — CryptoScamDB blacklist: failed ({e})')

    # Source 3: MyEtherWallet (MEW) ethereum-lists darklist
    try:
        r = requests.get(
            'https://raw.githubusercontent.com/MyEtherWallet/ethereum-lists/master/src/addresses/addresses-darklist.json',
            timeout=15)
        if r.status_code == 200:
            before = len(scam_address_set)
            for item in r.json():
                addr = item.get('address', '')
                if isinstance(addr, str) and addr.startswith('0x'):
                    scam_address_set.add(addr.lower())
            print(f'Source 3 — MEW darklist:           +{len(scam_address_set)-before:,} addresses')
        else:
            print(f'Source 3 — MEW darklist:           HTTP {r.status_code}, skipping')
    except Exception as e:
        print(f'Source 3 — MEW darklist:           failed ({e})')

    print(f'\nTotal unique scam addresses: {len(scam_address_set):,}')

    if len(scam_address_set) > 0:
        with open(SCAM_PATH, 'w') as f:
            json.dump(sorted(scam_address_set), f)
        print(f'Saved: {SCAM_PATH}')
        print('Scam set ready — Cell 3 will load from this file.')
    else:
        print('WARNING: No addresses fetched. avg_shortest_path will be 99.0 for all rows.')
        print('This is acceptable for the FYP — continue with Cell 3.')


Fetching scam addresses from live sources...
Source 1 — CryptoScamDB API:       HTTP 502, skipping
Source 2 — CryptoScamDB blacklist: +2,777 addresses
Source 3 — MEW darklist:           +283 addresses

Total unique scam addresses: 3,060
Saved: /content/drive/MyDrive/PhishGuard_v2/data/scam_addresses.json
Scam set ready — Cell 3 will load from this file.


In [5]:
# Cell 3 — Constants, imports, scam set
import pandas as pd
import numpy as np
import json
import time
import os
import joblib
from collections import Counter, defaultdict

UNLIMITED   = 2**256 - 1
PASS1_CKPT  = f'{DRIVE_BASE}/data/checkpoints/wallet_pass1_checkpoint.json'
PASS2_CKPT  = f'{DRIVE_BASE}/data/checkpoints/wallet_pass2_checkpoint.json'
OUTPUT_PATH = f'{DRIVE_BASE}/features/wallet_features.csv'
SCHEMA_PATH = f'{DRIVE_BASE}/models/wallet_feature_schema.json'

FEATURE_COLS = [
    'wallet_age_days', 'tx_count_in', 'tx_count_out', 'tx_count_total',
    'unique_counterparties_lifetime', 'fan_in_ratio', 'median_inter_tx_minutes',
    'std_inter_tx_minutes', 'avg_out_value_eth', 'std_out_value_eth',
    'min_in_value_eth', 'current_eth_balance', 'approval_count_total',
    'unlimited_approval_count_lifetime', 'unique_spenders_lifetime',
    'approval_concentration_top_spender', 'unlimited_approval_rate',
    'token_transfer_count', 'cross_token_approval_same_spender_ratio',
    'pagerank_subgraph', 'clustering_coefficient', 'reciprocity_ratio',
    'avg_shortest_path_to_known_scam'
]

scam_address_set = load_scam_set(DRIVE_BASE)
print('Cell 3 ready.')



Scam set loaded: 3,060 addresses
Cell 3 ready.


In [6]:
# Cell 4 — Define helper functions
def parse_approval_amount(log):
    topics = log.get('topics', [])
    data   = log.get('data', '0x')
    try:
        if len(topics) == 3:
            return int(data, 16) if data and data != '0x' else 0
        elif len(topics) == 4:
            return int(topics[3], 16) if topics[3] else 0
        return 0
    except Exception:
        return 0

def safe_divide(num, denom, default=0.0):
    return num / denom if denom != 0 else default

def safe_json(val):
    try:
        result = json.loads(val) if isinstance(val, str) else val
        return result if isinstance(result, list) else []
    except Exception:
        return []

print('Cell 4 helpers defined.')



Cell 4 helpers defined.


In [7]:
# Cell 5 — Define extract_wallet_features_pass1(row)
def extract_wallet_features_pass1(row):
    zeros = {
        'wallet_age_days': 0, 'tx_count_in': 0, 'tx_count_out': 0,
        'tx_count_total': 0, 'unique_counterparties_lifetime': 0,
        'fan_in_ratio': 0.5, 'median_inter_tx_minutes': 0,
        'std_inter_tx_minutes': 0, 'avg_out_value_eth': 0,
        'std_out_value_eth': 0, 'min_in_value_eth': 0,
        'current_eth_balance': 0, 'approval_count_total': 0,
        'unlimited_approval_count_lifetime': 0, 'unique_spenders_lifetime': 0,
        'approval_concentration_top_spender': 0, 'unlimited_approval_rate': 0,
        'token_transfer_count': 0, 'cross_token_approval_same_spender_ratio': 0
    }
    try:
        addr     = row['address'].lower()
        txs      = safe_json(row['txs_json'])
        tokentxs = safe_json(row['tokentxs_json'])
        logs     = safe_json(row['approval_logs_json'])
        now      = time.time()

        # Group A
        wallet_age_days = (now - int(txs[-1]['timeStamp'])) / 86400 if txs else 0
        tx_count_in     = sum(1 for t in txs if t.get('to',   '').lower() == addr)
        tx_count_out    = sum(1 for t in txs if t.get('from', '').lower() == addr)
        tx_count_total  = len(txs)

        counterparties = set()
        for t in txs:
            if t.get('from', '').lower() != addr:
                counterparties.add(t.get('from', '').lower())
            if t.get('to', '').lower() != addr:
                counterparties.add(t.get('to', '').lower())
        counterparties.discard('')
        unique_counterparties_lifetime = len(counterparties)

        fan_in_ratio = safe_divide(tx_count_in, tx_count_in + tx_count_out, 0.5)

        timestamps = sorted([int(t['timeStamp']) for t in txs if 'timeStamp' in t])
        if len(timestamps) >= 2:
            diffs = np.diff(timestamps)
            median_inter_tx_minutes = float(np.median(diffs)) / 60
            std_inter_tx_minutes    = float(np.std(diffs, ddof=0)) / 60
        else:
            median_inter_tx_minutes = 0
            std_inter_tx_minutes    = 0

        # Group B
        out_vals = [int(t.get('value', 0)) / 1e18
                    for t in txs
                    if t.get('from', '').lower() == addr
                    and t.get('isError') == '0'
                    and int(t.get('value', 0)) > 0]
        avg_out_value_eth = float(np.mean(out_vals))        if out_vals            else 0
        std_out_value_eth = float(np.std(out_vals, ddof=0)) if len(out_vals) >= 2  else 0

        in_vals = [int(t.get('value', 0)) / 1e18
                   for t in txs
                   if t.get('to', '').lower() == addr
                   and t.get('isError') == '0']
        min_in_value_eth    = float(min(in_vals)) if in_vals else 0
        current_eth_balance = float(row['balance_wei']) / 1e18

        # Group C
        approval_count_total = len(logs)

        unlimited_count = sum(1 for l in logs if parse_approval_amount(l) == UNLIMITED)
        unlimited_approval_count_lifetime = unlimited_count

        spenders = ['0x' + l['topics'][2][-40:]
                    for l in logs if len(l.get('topics', [])) >= 3]
        unique_spenders_lifetime = len(set(spenders))

        if approval_count_total > 0:
            spender_counts = Counter(spenders)
            approval_concentration_top_spender = safe_divide(
                spender_counts.most_common(1)[0][1], approval_count_total)
        else:
            approval_concentration_top_spender = 0

        unlimited_approval_rate = safe_divide(
            unlimited_approval_count_lifetime, approval_count_total)

        token_transfer_count = len(tokentxs)

        if unique_spenders_lifetime > 0:
            spender_tokens = defaultdict(set)
            for l in logs:
                if len(l.get('topics', [])) >= 3:
                    spender = '0x' + l['topics'][2][-40:]
                    token   = l.get('address', '').lower()
                    spender_tokens[spender].add(token)
            multi_token_spenders = sum(
                1 for tokens in spender_tokens.values() if len(tokens) > 1)
            cross_token_approval_same_spender_ratio = safe_divide(
                multi_token_spenders, unique_spenders_lifetime)
        else:
            cross_token_approval_same_spender_ratio = 0

        return {
            'wallet_age_days':                         wallet_age_days,
            'tx_count_in':                             tx_count_in,
            'tx_count_out':                            tx_count_out,
            'tx_count_total':                          tx_count_total,
            'unique_counterparties_lifetime':          unique_counterparties_lifetime,
            'fan_in_ratio':                            fan_in_ratio,
            'median_inter_tx_minutes':                 median_inter_tx_minutes,
            'std_inter_tx_minutes':                    std_inter_tx_minutes,
            'avg_out_value_eth':                       avg_out_value_eth,
            'std_out_value_eth':                       std_out_value_eth,
            'min_in_value_eth':                        min_in_value_eth,
            'current_eth_balance':                     current_eth_balance,
            'approval_count_total':                    approval_count_total,
            'unlimited_approval_count_lifetime':       unlimited_approval_count_lifetime,
            'unique_spenders_lifetime':                unique_spenders_lifetime,
            'approval_concentration_top_spender':      approval_concentration_top_spender,
            'unlimited_approval_rate':                 unlimited_approval_rate,
            'token_transfer_count':                    token_transfer_count,
            'cross_token_approval_same_spender_ratio': cross_token_approval_same_spender_ratio
        }
    except Exception:
        return zeros

print('Cell 5 pass1 extractor defined.')


Cell 5 pass1 extractor defined.


In [8]:
# Cell 6 — PASS 1: Extract features 1-19 with checkpoint
df = pd.read_csv(f'{DRIVE_BASE}/data/raw_wallet_data.csv')

if os.path.exists(PASS1_CKPT):
    with open(PASS1_CKPT) as f:
        pass1_results = json.load(f)
    done_addrs = set(r['address'] for r in pass1_results)
    print(f'Resumed Pass 1: {len(pass1_results)} done')
else:
    pass1_results = []
    done_addrs    = set()
    print('Starting Pass 1 fresh')

remaining = df[~df['address'].isin(done_addrs)]
print(f'Rows remaining: {len(remaining)}')
errors_p1 = []

for i, (_, row) in enumerate(remaining.iterrows()):
    try:
        feats            = extract_wallet_features_pass1(row)
        feats['address'] = row['address']
        feats['label']   = row['label']
        pass1_results.append(feats)
    except Exception as e:
        errors_p1.append({'address': row['address'], 'error': str(e)})

    if (i + 1) % 100 == 0:
        with open(PASS1_CKPT, 'w') as f:
            json.dump(pass1_results, f)
        print(f'  Checkpoint: {len(pass1_results)} done | {len(errors_p1)} errors')

with open(PASS1_CKPT, 'w') as f:
    json.dump(pass1_results, f)
print(f'Pass 1 complete: {len(pass1_results)} rows | {len(errors_p1)} errors')


Starting Pass 1 fresh
Rows remaining: 4000
  Checkpoint: 100 done | 0 errors
  Checkpoint: 200 done | 0 errors
  Checkpoint: 300 done | 0 errors
  Checkpoint: 400 done | 0 errors
  Checkpoint: 500 done | 0 errors
  Checkpoint: 600 done | 0 errors
  Checkpoint: 700 done | 0 errors
  Checkpoint: 800 done | 0 errors
  Checkpoint: 900 done | 0 errors
  Checkpoint: 1000 done | 0 errors
  Checkpoint: 1100 done | 0 errors
  Checkpoint: 1200 done | 0 errors
  Checkpoint: 1300 done | 0 errors
  Checkpoint: 1400 done | 0 errors
  Checkpoint: 1500 done | 0 errors
  Checkpoint: 1600 done | 0 errors
  Checkpoint: 1700 done | 0 errors
  Checkpoint: 1800 done | 0 errors
  Checkpoint: 1900 done | 0 errors
  Checkpoint: 2000 done | 0 errors
  Checkpoint: 2100 done | 0 errors
  Checkpoint: 2200 done | 0 errors
  Checkpoint: 2300 done | 0 errors
  Checkpoint: 2400 done | 0 errors
  Checkpoint: 2500 done | 0 errors
  Checkpoint: 2600 done | 0 errors
  Checkpoint: 2700 done | 0 errors
  Checkpoint: 2800 do

In [9]:
# Cell 7 — PASS 2: Extract graph features 20-23 with checkpoint
with open(PASS1_CKPT) as f:
    pass1_results = json.load(f)
df_pass1 = pd.DataFrame(pass1_results)

df_raw       = pd.read_csv(f'{DRIVE_BASE}/data/raw_wallet_data.csv')
df_raw.index = df_raw['address'].str.lower()

if os.path.exists(PASS2_CKPT):
    with open(PASS2_CKPT) as f:
        pass2_results = json.load(f)
    done_addrs_p2 = set(r['address'] for r in pass2_results)
    print(f'Resumed Pass 2: {len(pass2_results)} done')
else:
    pass2_results = []
    done_addrs_p2 = set()
    print('Starting Pass 2 fresh')

remaining_p2 = df_pass1[~df_pass1['address'].isin(done_addrs_p2)]
print(f'Rows remaining for graph extraction: {len(remaining_p2)}')
errors_p2 = []

for i, (_, row) in enumerate(remaining_p2.iterrows()):
    addr = row['address'].lower()
    try:
        raw_row = df_raw.loc[addr]
        txs     = safe_json(raw_row['txs_json'])
        gfeats  = extract_graph_features(txs, addr, scam_address_set)
        gfeats['address'] = addr
        pass2_results.append(gfeats)
    except Exception as e:
        errors_p2.append({'address': addr, 'error': str(e)})
        pass2_results.append({
            'address':                         addr,
            'pagerank_subgraph':               0.0,
            'clustering_coefficient':          0.0,
            'reciprocity_ratio':               0.0,
            'avg_shortest_path_to_known_scam': 99.0
        })

    if (i + 1) % 100 == 0:
        with open(PASS2_CKPT, 'w') as f:
            json.dump(pass2_results, f)
        print(f'  Checkpoint: {len(pass2_results)} done | {len(errors_p2)} errors')

with open(PASS2_CKPT, 'w') as f:
    json.dump(pass2_results, f)
print(f'Pass 2 complete: {len(pass2_results)} rows | {len(errors_p2)} errors')


Starting Pass 2 fresh
Rows remaining for graph extraction: 4000
  Checkpoint: 100 done | 0 errors
  Checkpoint: 200 done | 0 errors
  Checkpoint: 300 done | 0 errors
  Checkpoint: 400 done | 0 errors
  Checkpoint: 500 done | 0 errors
  Checkpoint: 600 done | 0 errors
  Checkpoint: 700 done | 0 errors
  Checkpoint: 800 done | 0 errors
  Checkpoint: 900 done | 0 errors
  Checkpoint: 1000 done | 0 errors
  Checkpoint: 1100 done | 0 errors
  Checkpoint: 1200 done | 0 errors
  Checkpoint: 1300 done | 0 errors
  Checkpoint: 1400 done | 0 errors
  Checkpoint: 1500 done | 0 errors
  Checkpoint: 1600 done | 0 errors
  Checkpoint: 1700 done | 0 errors
  Checkpoint: 1800 done | 0 errors
  Checkpoint: 1900 done | 0 errors
  Checkpoint: 2000 done | 0 errors
  Checkpoint: 2100 done | 0 errors
  Checkpoint: 2200 done | 0 errors
  Checkpoint: 2300 done | 0 errors
  Checkpoint: 2400 done | 0 errors
  Checkpoint: 2500 done | 0 errors
  Checkpoint: 2600 done | 0 errors
  Checkpoint: 2700 done | 0 errors


In [11]:
# Cell 8 — Merge, clean, validate
with open(PASS1_CKPT) as f:
    pass1_results = json.load(f)
with open(PASS2_CKPT) as f:
    pass2_results = json.load(f)

df_p1 = pd.DataFrame(pass1_results)
df_p2 = pd.DataFrame(pass2_results)

df_p2 = df_p2.drop(columns=['label'], errors='ignore')

df_features = df_p1.merge(df_p2, on='address', how='left')
df_features = df_features[['address', 'label'] + FEATURE_COLS]
df_features = df_features.replace([float('inf'), float('-inf')], 0)
df_features = df_features.fillna(0)

print('=== VALIDATION ===')
assert df_features.shape == (4000, 25),                'Wrong shape'
assert df_features['label'].value_counts()[1] == 2000, 'Phishing count wrong'
assert df_features['label'].value_counts()[0] == 2000, 'Benign count wrong'
assert df_features.isnull().sum().sum() == 0,          'Nulls found'
assert not np.isinf(df_features[FEATURE_COLS].values).any(), 'Inf values found'
for col in FEATURE_COLS:
    assert pd.api.types.is_numeric_dtype(df_features[col]), f'{col} not numeric'

age_ok   = (df_features['wallet_age_days'] > 0).mean()
ulim_max = df_features['unlimited_approval_count_lifetime'].max()
scam_pct = (df_features['avg_shortest_path_to_known_scam'] == 99.0).mean()
print(f'wallet_age_days > 0:       {age_ok:.1%} (expect >90%)')
print(f'unlimited_approval max:    {ulim_max} (expect >0)')
print(f'avg_shortest_path == 99.0: {scam_pct:.1%} (high is ok if scam set small)')
print(f'Shape: {df_features.shape}')
print('All validation checks passed.')


=== VALIDATION ===
wallet_age_days > 0:       100.0% (expect >90%)
unlimited_approval max:    335 (expect >0)
avg_shortest_path == 99.0: 94.0% (high is ok if scam set small)
Shape: (4000, 25)
All validation checks passed.


In [12]:
# Cell 9 — Save outputs
df_features.to_csv(OUTPUT_PATH, index=False)
with open(SCHEMA_PATH, 'w') as f:
    json.dump(FEATURE_COLS, f)
print(f'Saved: {OUTPUT_PATH}')
print(f'Saved: {SCHEMA_PATH}')

df_check = pd.read_csv(OUTPUT_PATH)
with open(SCHEMA_PATH) as f:
    schema_check = json.load(f)

assert df_check.shape    == (4000, 25), f'Reload shape wrong: {df_check.shape}'
assert len(schema_check) == 23,         f'Schema length wrong: {len(schema_check)}'
print(f'Verified CSV shape: {df_check.shape}')
print(f'Verified schema items: {len(schema_check)}')
print('Notebook 02 complete.')


Saved: /content/drive/MyDrive/PhishGuard_v2/features/wallet_features.csv
Saved: /content/drive/MyDrive/PhishGuard_v2/models/wallet_feature_schema.json
Verified CSV shape: (4000, 25)
Verified schema items: 23
Notebook 02 complete.


In [13]:
# Spot-check: verify features for a specific row
import json

def inspect_wallet(address):
    # Load the saved CSV
    row_feat = df_features[df_features['address'].str.lower() == address.lower()].iloc[0]
    
    # Load raw data for the same address
    df_raw = pd.read_csv(f'{DRIVE_BASE}/data/raw_wallet_data.csv')
    row_raw = df_raw[df_raw['address'].str.lower() == address.lower()].iloc[0]
    
    txs      = safe_json(row_raw['txs_json'])
    logs     = safe_json(row_raw['approval_logs_json'])
    tokentxs = safe_json(row_raw['tokentxs_json'])
    
    print(f'=== {address} ===')
    print(f'Label:              {"PHISHING" if row_feat["label"]==1 else "BENIGN"}')
    print()
    print(f'--- Raw counts ---')
    print(f'txs_json rows:      {len(txs)}')
    print(f'approval_logs rows: {len(logs)}')
    print(f'tokentxs rows:      {len(tokentxs)}')
    print()
    print(f'--- Extracted features ---')
    for col in FEATURE_COLS:
        print(f'  {col:<45} {row_feat[col]}')

# Check one phishing and one benign wallet
phishing_addr = df_features[df_features['label']==1].iloc[0]['address']
benign_addr   = df_features[df_features['label']==0].iloc[0]['address']

inspect_wallet(phishing_addr)
print()
inspect_wallet(benign_addr)


=== 0x000000000532b45f47779fce440748893b257865 ===
Label:              PHISHING

--- Raw counts ---
txs_json rows:      23
approval_logs rows: 0
tokentxs rows:      0

--- Extracted features ---
  wallet_age_days                               2076.0479425032877
  tx_count_in                                   20
  tx_count_out                                  3
  tx_count_total                                23
  unique_counterparties_lifetime                23
  fan_in_ratio                                  0.8695652173913043
  median_inter_tx_minutes                       11.716666666666667
  std_inter_tx_minutes                          172.45102810432252
  avg_out_value_eth                             39.04315781613226
  std_out_value_eth                             0.001191027847765349
  min_in_value_eth                              0.000409938396784603
  current_eth_balance                           0.0
  approval_count_total                          0
  unlimited_approval_count_l